In [1]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, average_precision_score, f1_score, 
    precision_recall_curve, roc_curve, roc_auc_score,
    accuracy_score, precision_score, recall_score, confusion_matrix
)
from codecarbon import EmissionsTracker

# ====== CONFIG ======
IN_FILE = "risultati_merged_mixed_enriched.csv"
MODEL_OUT = "catboost_td_cheap.cbm"
META_OUT = "catboost_td_cheap_meta.json"
PLOTS_DIR = "plots_catboost_cheap"
TEXT_COL = "orig_text"
TARGET_COL = "logit_td"
LABEL_COL = "label"
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Crea directory per i grafici
os.makedirs(PLOTS_DIR, exist_ok=True)

# ====== LOAD ======
print("Caricamento dataset...")
df = pd.read_csv(IN_FILE)
df.columns = [c.strip() for c in df.columns]

# sanity checks
required = [TEXT_COL, TARGET_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Mancano colonne richieste: {missing}. Colonne disponibili: {df.columns.tolist()}")

# pulizia base
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").str.strip()
df = df[df[TEXT_COL].str.len() > 0].copy()

# target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()

print(f"Dataset caricato: {len(df)} righe")

# ====== ESTRAZIONE CHEAP FEATURES ======
print("Estrazione cheap features...")

def extract_cheap_features(text):
    """Estrae feature semplici e veloci da calcolare senza NLP"""
    features = {}
    
    # Lunghezze
    features['char_length'] = len(text)
    features['word_count'] = len(text.split())
    features['avg_word_length'] = features['char_length'] / max(features['word_count'], 1)
    
    # Conteggi caratteri speciali
    features['num_uppercase'] = sum(1 for c in text if c.isupper())
    features['num_digits'] = sum(1 for c in text if c.isdigit())
    features['num_spaces'] = text.count(' ')
    features['num_punctuation'] = sum(1 for c in text if c in '.,;:!?')
    features['num_special_chars'] = sum(1 for c in text if not c.isalnum() and not c.isspace())
    
    # Ratios
    features['uppercase_ratio'] = features['num_uppercase'] / max(features['char_length'], 1)
    features['digit_ratio'] = features['num_digits'] / max(features['char_length'], 1)
    features['punctuation_ratio'] = features['num_punctuation'] / max(features['char_length'], 1)
    features['special_char_ratio'] = features['num_special_chars'] / max(features['char_length'], 1)
    
    # Conteggi newline e tab
    features['num_newlines'] = text.count('\n')
    features['num_tabs'] = text.count('\t')
    
    # Parole uniche
    words = text.lower().split()
    features['unique_word_count'] = len(set(words))
    features['unique_word_ratio'] = features['unique_word_count'] / max(features['word_count'], 1)
    
    # Keyword TD-related (cheap string matching)
    td_keywords = ['debt', 'refactor', 'todo', 'fixme', 'hack', 'workaround', 'technical', 'issue']
    text_lower = text.lower()
    features['num_td_keywords'] = sum(1 for kw in td_keywords if kw in text_lower)
    features['has_todo'] = int('todo' in text_lower or 'fixme' in text_lower)
    features['has_debt'] = int('debt' in text_lower)
    
    # Simboli di codice comuni
    code_symbols = ['{', '}', '(', ')', '[', ']', ';', '=', '+', '-', '*', '/']
    features['num_code_symbols'] = sum(text.count(sym) for sym in code_symbols)
    features['code_symbol_ratio'] = features['num_code_symbols'] / max(features['char_length'], 1)
    
    # Presenza di URL o path
    features['has_url'] = int(bool(re.search(r'https?://|www\.', text)))
    features['has_path'] = int(bool(re.search(r'[/\\][a-zA-Z0-9_]+', text)))
    
    return features

# Applica estrazione feature
print("Estrazione in corso...")
cheap_features = df[TEXT_COL].apply(extract_cheap_features).apply(pd.Series)
df = pd.concat([df, cheap_features], axis=1)

# Feature columns (solo numeriche, NO testo)
num_cols = list(cheap_features.columns)
feature_cols = num_cols.copy()

print(f"Cheap features estratte: {len(feature_cols)}")
print(f"Feature list: {feature_cols[:10]}...")

# Gestione NaN/Inf
for col in feature_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].fillna(0.0)

# ====== SPLIT ======
print("Split train/validation...")
train_df, valid_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[LABEL_COL] if LABEL_COL in df.columns else None
)

print(f"Train: {len(train_df)} | Validation: {len(valid_df)}")

# IMPORTANTE: NO text_features, solo numeriche
train_pool = Pool(
    data=train_df[feature_cols],
    label=train_df[TARGET_COL]
)
valid_pool = Pool(
    data=valid_df[feature_cols],
    label=valid_df[TARGET_COL]
)

# ====== TRAIN CON CARBON TRACKING ======
print("\nInizio training con carbon tracking...")
tracker = EmissionsTracker(project_name="catboost_cheap", output_dir=PLOTS_DIR)
tracker.start()

start_time = time.time()

model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5,
    random_seed=RANDOM_STATE,
    early_stopping_rounds=200,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

training_time = time.time() - start_time
emissions = tracker.stop()

print(f"\nTraining completato in {training_time:.2f} secondi ({training_time/60:.2f} minuti)")
print(f"Emissioni CO2: {emissions:.6f} kg")

# ====== PREDICTIONS ======
print("\nGenerazione predizioni...")
pred_logit_train = model.predict(train_pool)
pred_logit = model.predict(valid_pool)

# ====== EVAL (distillation fidelity) ======
rmse_train = np.sqrt(mean_squared_error(train_df[TARGET_COL].values, pred_logit_train))
rmse = np.sqrt(mean_squared_error(valid_df[TARGET_COL].values, pred_logit))
print(f"\nRMSE(logit) train: {rmse_train:.6f}")
print(f"RMSE(logit) valid: {rmse:.6f}")

# ====== EVAL (classification vs label) ======
p_hat_train = 1.0 / (1.0 + np.exp(-pred_logit_train))
p_hat = 1.0 / (1.0 + np.exp(-pred_logit))

metrics = {}

if LABEL_COL in valid_df.columns and valid_df[LABEL_COL].notna().any():
    y_train = train_df[LABEL_COL].astype(int).values
    y_true = valid_df[LABEL_COL].astype(int).values
    
    # ROC-AUC
    roc_auc_train = roc_auc_score(y_train, p_hat_train)
    roc_auc = roc_auc_score(y_true, p_hat)
    print(f"\nROC-AUC train: {roc_auc_train:.6f}")
    print(f"ROC-AUC valid: {roc_auc:.6f}")
    
    # PR-AUC
    pr_auc_train = average_precision_score(y_train, p_hat_train)
    pr_auc = average_precision_score(y_true, p_hat)
    print(f"PR-AUC train: {pr_auc_train:.6f}")
    print(f"PR-AUC valid: {pr_auc:.6f}")
    
    # Trova soglia ottima per F1 su validation
    precisions, recalls, thresholds = precision_recall_curve(y_true, p_hat)
    f1s = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    best_thr = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5
    
    y_pred = (p_hat >= best_thr).astype(int)
    
    # Metriche di classificazione
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"\nSoglia ottimale (F1): {best_thr:.6f}")
    print(f"Accuracy: {accuracy:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall: {recall:.6f}")
    print(f"F1-Score: {f1:.6f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(f"\nConfusion Matrix:")
    print(cm)
    
    metrics = {
        "roc_auc_train": roc_auc_train,
        "roc_auc_valid": roc_auc,
        "pr_auc_train": pr_auc_train,
        "pr_auc_valid": pr_auc,
        "best_threshold": best_thr,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }
    
    # ====== PLOT ROC CURVE ======
    fpr, tpr, _ = roc_curve(y_true, p_hat)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve - CatBoost Cheap Features', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "roc_curve.png"), dpi=300)
    print(f"\nROC curve salvata in: {os.path.join(PLOTS_DIR, 'roc_curve.png')}")
    plt.close()
    
    # ====== PLOT PR CURVE ======
    plt.figure(figsize=(8, 6))
    plt.plot(recalls, precisions, linewidth=2, label=f'PR curve (AUC = {pr_auc:.4f})')
    plt.axhline(y=y_true.mean(), color='k', linestyle='--', linewidth=1, label=f'Baseline ({y_true.mean():.4f})')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title('Precision-Recall Curve - CatBoost Cheap Features', fontsize=14, fontweight='bold')
    plt.legend(loc="lower left", fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "pr_curve.png"), dpi=300)
    print(f"PR curve salvata in: {os.path.join(PLOTS_DIR, 'pr_curve.png')}")
    plt.close()
    
    # ====== PLOT CONFUSION MATRIX ======
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['Non-TD', 'TD'], yticklabels=['Non-TD', 'TD'])
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Confusion Matrix - CatBoost Cheap Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "confusion_matrix.png"), dpi=300)
    print(f"Confusion matrix salvata in: {os.path.join(PLOTS_DIR, 'confusion_matrix.png')}")
    plt.close()
    
    # ====== PLOT DISTRIBUTION OF PREDICTIONS ======
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribuzione per classe vera
    ax1.hist(p_hat[y_true == 0], bins=50, alpha=0.6, label='Non-TD (0)', color='blue')
    ax1.hist(p_hat[y_true == 1], bins=50, alpha=0.6, label='TD (1)', color='red')
    ax1.axvline(best_thr, color='green', linestyle='--', linewidth=2, label=f'Soglia ottimale ({best_thr:.3f})')
    ax1.set_xlabel('Predicted Probability', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.set_title('Distribution of Predicted Probabilities', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(alpha=0.3)
    
    # Logit distribution
    ax2.hist(pred_logit[y_true == 0], bins=50, alpha=0.6, label='Non-TD (0)', color='blue')
    ax2.hist(pred_logit[y_true == 1], bins=50, alpha=0.6, label='TD (1)', color='red')
    ax2.set_xlabel('Predicted Logit', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.set_title('Distribution of Predicted Logits', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "prediction_distributions.png"), dpi=300)
    print(f"Distribution plots salvate in: {os.path.join(PLOTS_DIR, 'prediction_distributions.png')}")
    plt.close()
    
else:
    print("Label non presente -> salto metriche classificazione.")

# ====== FEATURE IMPORTANCE ======
try:
    feature_importance = model.get_feature_importance(train_pool)
    feature_names = feature_cols
    
    if len(feature_importance) > 0:
        # Crea dataframe con importanze
        fi_df = pd.DataFrame({
            'feature': feature_names[:len(feature_importance)],
            'importance': feature_importance
        }).sort_values('importance', ascending=False)
        
        print(f"\nTop {min(20, len(fi_df))} Feature Importance:")
        print(fi_df.head(20).to_string(index=False))
        
        # Plot top features
        top_n = min(20, len(fi_df))
        plt.figure(figsize=(10, max(6, top_n * 0.3)))
        plt.barh(range(top_n), fi_df['importance'].head(top_n), color='steelblue')
        plt.yticks(range(top_n), fi_df['feature'].head(top_n))
        plt.xlabel('Importance', fontsize=12)
        plt.ylabel('Feature', fontsize=12)
        plt.title(f'Top {top_n} Cheap Feature Importance', fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, "feature_importance.png"), dpi=300)
        print(f"\nFeature importance plot salvato in: {os.path.join(PLOTS_DIR, 'feature_importance.png')}")
        plt.close()
        
        # Salva feature importance in CSV
        fi_df.to_csv(os.path.join(PLOTS_DIR, "feature_importance.csv"), index=False)
        print(f"Feature importance CSV salvato in: {os.path.join(PLOTS_DIR, 'feature_importance.csv')}")
        
except Exception as e:
    print(f"\nNote: Impossibile calcolare feature importance: {e}")

# ====== SAVE MODEL ======
model.save_model(MODEL_OUT)

# ====== SAVE METADATA ======
meta = {
    "model_type": "cheap_features",
    "description": "Modello basato su cheap features (feature semplici e veloci da calcolare)",
    "input_file": IN_FILE,
    "text_col": TEXT_COL,
    "target_col": TARGET_COL,
    "label_col": LABEL_COL if LABEL_COL in df.columns else None,
    "cheap_features": num_cols,
    "num_features": len(feature_cols),
    "feature_cols": feature_cols,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "dataset_size": len(df),
    "train_size": len(train_df),
    "valid_size": len(valid_df),
    "rmse_train_logit": float(rmse_train),
    "rmse_valid_logit": float(rmse),
    "training_time_seconds": training_time,
    "training_time_minutes": training_time / 60,
    "co2_emissions_kg": float(emissions),
    "model_iterations": model.get_best_iteration(),
    "model_file": MODEL_OUT,
    **metrics
}

with open(META_OUT, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("\n" + "="*60)
print("RIEPILOGO FINALE - MODELLO CHEAP FEATURES")
print("="*60)
print(f"Modello salvato: {MODEL_OUT}")
print(f"Metadata salvati: {META_OUT}")
print(f"Grafici salvati in: {PLOTS_DIR}/")
print(f"Numero cheap features: {len(feature_cols)}")
print(f"Tempo training: {training_time:.2f}s ({training_time/60:.2f} min)")
print(f"CO2 emissions: {emissions:.6f} kg")
print(f"RMSE validation: {rmse:.6f}")
if metrics:
    print(f"ROC-AUC validation: {metrics.get('roc_auc_valid', 'N/A'):.6f}")
    print(f"PR-AUC validation: {metrics.get('pr_auc_valid', 'N/A'):.6f}")
    print(f"F1-Score validation: {metrics.get('f1_score', 'N/A'):.6f}")
print("="*60)


Caricamento dataset...
Dataset caricato: 20828 righe
Estrazione cheap features...
Estrazione in corso...


[codecarbon WARNING @ 21:49:06] Multiple instances of codecarbon are allowed to run at the same time.


Cheap features estratte: 23
Feature list: ['char_length', 'word_count', 'avg_word_length', 'num_uppercase', 'num_digits', 'num_spaces', 'num_punctuation', 'num_special_chars', 'uppercase_ratio', 'digit_ratio']...
Split train/validation...
Train: 16662 | Validation: 4166

Inizio training con carbon tracking...


[codecarbon INFO @ 21:49:10] [setup] RAM Tracking...
[codecarbon INFO @ 21:49:10] [setup] CPU Tracking...
[codecarbon WARNING @ 21:49:11] We saw that you have a 13th Gen Intel(R) Core(TM) i7-13620H but we don't know it. Please contact us.
[codecarbon WARNING @ 21:49:11] We will use the default power consumption of 4 W per thread for your 16 CPU, so 64W.
[codecarbon WARNING @ 21:49:11] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 21:49:11] CPU Model on constant consumption mode: 13th Gen Intel(R) Core(TM) i7-13620H
[codecarbon WARNING @ 21:49:11] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 21:49:11] [setup] GPU Tracking...
[codecarbon INFO @ 21:49:11] No GPU found.
[codecarbon INFO @ 21:49:11] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Me

0:	learn: 4.1059386	test: 4.1037652	best: 4.1037652 (0)	total: 138ms	remaining: 11m 31s
200:	learn: 3.7037374	test: 3.7830146	best: 3.7830146 (200)	total: 996ms	remaining: 23.8s
400:	learn: 3.6289799	test: 3.7770422	best: 3.7768262 (399)	total: 1.88s	remaining: 21.5s
600:	learn: 3.5632637	test: 3.7768794	best: 3.7754950 (477)	total: 2.76s	remaining: 20.2s


[codecarbon INFO @ 21:49:19] Energy consumed for RAM : 0.000012 kWh. RAM Power : 10.0 W


Stopped by overfitting detector  (200 iterations wait)

bestTest = 3.77549498
bestIteration = 477

Shrink model to first 478 iterations.


[codecarbon INFO @ 21:49:19] Delta energy consumed for CPU with cpu_load : 0.000020 kWh, power : 16.783222873600007 W
[codecarbon INFO @ 21:49:19] Energy consumed for All CPU : 0.000020 kWh
[codecarbon INFO @ 21:49:19] 0.000032 kWh of electricity and 0.000000 L of water were used since the beginning.



Training completato in 3.28 secondi (0.05 minuti)
Emissioni CO2: 0.000011 kg

Generazione predizioni...

RMSE(logit) train: 3.603769
RMSE(logit) valid: 3.775495

ROC-AUC train: 0.768458
ROC-AUC valid: 0.706890
PR-AUC train: 0.783342
PR-AUC valid: 0.736825

Soglia ottimale (F1): 0.128249
Accuracy: 0.568891
Precision: 0.537897
Recall: 0.954217
F1-Score: 0.687978

Confusion Matrix:
[[ 390 1701]
 [  95 1980]]

ROC curve salvata in: plots_catboost_cheap\roc_curve.png
PR curve salvata in: plots_catboost_cheap\pr_curve.png
Confusion matrix salvata in: plots_catboost_cheap\confusion_matrix.png
Distribution plots salvate in: plots_catboost_cheap\prediction_distributions.png

Top 20 Feature Importance:
           feature  importance
   avg_word_length   19.776605
 unique_word_ratio   18.163781
 unique_word_count   11.694549
       char_length   10.991298
   num_td_keywords    9.635689
          has_debt    7.603131
        word_count    6.733345
        num_spaces    6.534840
special_char_ratio